# STM - Superior Tribunal Militar

Example notebook for the STM scraper, which searches the court's decisions in the [Consulta Judicial Unificada da Justiça Militar da União](https://jurisprudencia.stm.jus.br/).

| Feature | Available |
|---------|----------|
| cjsg    | Yes      |

The portal sits behind a WAF that refuses requests depending on the source IP address: it rejected every request from a datacenter IP address outside Brazil and accepted a residential one in Brazil (16 November 2026). When refused, `cjsg` raises `BotChallengeBlockedError` instead of returning an empty search. Run this notebook from a connection the portal accepts.

In [ ]:
import juscraper as jus

stm = jus.scraper("stm")

## Basic search

Each page has 100 decisions, from the most recent judgment to the oldest. `paginas` is 1-based; `None` (default) downloads every page.

In [ ]:
df = stm.cjsg("deserção", paginas=1)

print(df.shape)
df.head(3)

## Available columns

In [ ]:
df.dtypes

In [ ]:
print(df["ementa"].iloc[0][:300])

## Search operators and fields

`pesquisa` takes the operators of the portal: `E` (the default between words), `OU`, `NÃO`, quotes for an exact phrase, and `*` and `?` as wildcards. `campo` chooses where the term is searched: `"ementa"` (default), `"inteiro_teor"` or `"numero"` (case number, complete or partial).

In [ ]:
df_ou = stm.cjsg("peculato OU furto", paginas=1)
df_frase = stm.cjsg('"corrupção ativa"', campo="inteiro_teor", paginas=1)

print(df_ou.shape, df_frase.shape)

In [ ]:
stm.cjsg("7000612-54.2022.7.00.0000", campo="numero").drop(columns="ementa")

## Filters

`classe`, `relator`, `relator_acordao`, `revisor` and `assunto` are the filters of the "Filtrar por" column of the portal. They match the exact value the portal shows, in upper case.

In [ ]:
df_apelacao = stm.cjsg("deserção", classe="APELAÇÃO", paginas=1)

print(df_apelacao.shape)
df_apelacao["classe"].value_counts()

## Date range

`data_julgamento_inicio`/`data_julgamento_fim` and `data_publicacao_inicio`/`data_publicacao_fim` take `DD/MM/YYYY`, `YYYY-MM-DD` or a `date`.

The portal filters by date range only in its advanced search, so with these filters the search goes through it. The advanced search looks for `pesquisa` as an exact phrase in the chosen `campo` and does not apply `E`, `OU`, `NÃO` or the wildcards. A term with more than one word outside quotes emits a `UserWarning`; put it in quotes to confirm the exact phrase.

In [ ]:
df_datas = stm.cjsg(
    "deserção",
    data_julgamento_inicio="2025-03-01",
    data_julgamento_fim="2025-03-31",
)

print(df_datas.shape)
print(df_datas["data_julgamento"].min(), df_datas["data_julgamento"].max())
df_datas.head(3)

An empty `pesquisa` returns everything in the range.

In [ ]:
df_semana = stm.cjsg("", data_julgamento_inicio="10/03/2025", data_julgamento_fim="13/03/2025")

print(df_semana.shape)
df_semana["classe"].value_counts().head()

## Download and parse separately

`cjsg_download` returns the raw HTML of each page and `cjsg_parse` turns it into the DataFrame.

In [ ]:
brutos = stm.cjsg_download("deserção", paginas=1)
df_parsed = stm.cjsg_parse(brutos)

print(len(brutos), df_parsed.shape)